# Logic converted from matlab file

In [ ]:
import os
import time
import numpy as np
import nibabel as nib
import scipy.io as sio
import h5py
import pickle
import gc
from google.colab import drive



# -----------------------------------------------------------------------
# Paths config – adjust if needed
# -----------------------------------------------------------------------
NSD_BASE = "/content/drive/MyDrive/V1_Drift/NSD"
PRFSAMPLE_DIR = "/content/drive/MyDrive/V1_Drift/prfsample_expand"
SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------
def load_nifti(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return nib.load(path).get_fdata()

def rsquared(resid, orig):
    """
    MATLAB:
        r2 = 1 - sum((Xresid).^2)/sum((Xorig - mean(Xorig)).^2);
    """
    resid = np.asarray(resid, dtype=np.float64)
    orig = np.asarray(orig, dtype=np.float64)
    num = np.sum(resid ** 2)
    denom = np.sum((orig - orig.mean()) ** 2)
    if denom <= 0:
        return np.nan
    return 1.0 - num / denom

def lstsq_coeff(X, y):
    """
    Wrapper for least-squares regression: MATLAB 'X \\ y'.
    X: (n_trials, n_predictors)
    y: (n_trials,)
    Returns: (n_predictors,)
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef.astype(np.float32)


# -----------------------------------------------------------------------
# Main function: regressPrfSplit_expand → Python
# -----------------------------------------------------------------------
def regress_prf_split_expand(isub, visualRegions=None, to_zscore=0):
    """
    Python version of regressPrfSplit_expand.m

    Args:
        isub (int): subject index 1..8
        visualRegions (int or list[int]): which visualRegion group(s) to run (1..4)
        to_zscore (int): normalization flag 0–4 (same meaning as MATLAB)
    """
    if visualRegions is None:
        visualRegions = [1]
    elif isinstance(visualRegions, int):
        visualRegions = [visualRegions]

    print(f"\n=== regress_prf_split_expand: sub {isub}, visualRegions {visualRegions}, to_zscore={to_zscore} ===")
    t0_global = time.time()

    # ---------------- NSD session counts ----------------
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]  # MATLAB nsessionsSub
    nsessions = nsessions_sub[isub - 1]
    nsplits = nsessions

    # ---------------- Paths ----------------
    betas_folder = os.path.join(NSD_BASE, f"subj{isub:02d}", "betas")
    nsd_expdesign_path = os.path.join(
        NSD_BASE, "nsddata", "experiments", "nsd", "nsd_expdesign.mat"
    )
    visual_rois_path = os.path.join(NSD_BASE, f"subj{isub:02d}", "func1pt8mm", "roi", "prf-visualrois.nii.gz")

    # ---------------- ROI labels ----------------
    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
    visRoiData = load_nifti(visual_rois_path).ravel()  # flatten like MATLAB visRoiData(:)

    # ---------------- Load NSD design ----------------
    nsd_design = sio.loadmat(nsd_expdesign_path)
    masterordering = nsd_design["masterordering"].flatten() - 1  # 0-based
    subjectim = nsd_design["subjectim"]           # shape (8, ntrials_per_subj)
    subDesign = subjectim[isub - 1, masterordering]  # trial → image ID

    # 30,000 trials total; trials per session = 10 runs × 75 trials
    ntrials = masterordering.size
    trials_per_session = 10 * 75  # 750

    # ---------------- Build splitImgTrials (nsplits × ntrials) ----------------
    splitImgTrials = np.zeros((nsplits, ntrials), dtype=np.int8)
    itrial = 0
    for isplit in range(nsplits):
        start = itrial
        end = itrial + trials_per_session
        splitImgTrials[isplit, start:end] = 1
        itrial += trials_per_session

    # ---------------- Precompute betas across sessions & ROIs ----------------
    # rois are 0-based here; visRoiData labels are 1-based (1..7), so mask with (roi+1)
    roi_indices = {1: [0, 1], 2: [2, 3], 3: [4, 5], 4: [6]}  # same grouping as MATLAB
    all_rois_needed = sorted({r for vr in visualRegions for r in roi_indices[vr]})

    # We'll build roiBetas as dict: roinum -> (nvox, ntrials_used)
    roi_betas_list = {roinum: [] for roinum in all_rois_needed}
    roiInd = {}   # roinum -> voxel indices in whole brain
    ntrials_total = 0

    print("\n-- Loading betas and building roiBetas --")
    for isession in range(1, nsessions + 1):
        # Watch the extension; change to '.nii' if your files are uncompressed
        betas_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii.gz")
        print(f"  Session {isession:02d}: {betas_file}")
        betas_4d = load_nifti(betas_file)            # (X, Y, Z, 750)
        betas_4d = betas_4d.astype(np.float32) / 300.0
        betas_2d = betas_4d.reshape(-1, betas_4d.shape[-1])  # (nvox_total, trials_per_session)
        del betas_4d
        gc.collect()

        for roinum in all_rois_needed:
            roi_mask = (visRoiData == (roinum + 1))
            roi_data = betas_2d[roi_mask, :]  # (nvox_roi, trials_per_session)

            if roinum not in roiInd:
                roiInd[roinum] = np.flatnonzero(roi_mask)

            if to_zscore == 1:
                # z-score per voxel across time
                m = roi_data.mean(axis=1, keepdims=True)
                s = roi_data.std(axis=1, ddof=0, keepdims=True)
                s[s == 0] = 1.0
                roi_norm = (roi_data - m) / s
            elif to_zscore == 2:
                # remove mean per voxel
                m = roi_data.mean(axis=1, keepdims=True)
                roi_norm = roi_data - m
            elif to_zscore == 3:
                # subtract mean, zscore, add back mean
                m = roi_data.mean(axis=1, keepdims=True)
                centered = roi_data - m
                s = centered.std(axis=1, ddof=0, keepdims=True)
                s[s == 0] = 1.0
                roi_norm = m + centered / s
            elif to_zscore == 4:
                # remove ROI grand mean
                grand_mean = roi_data.mean()
                roi_norm = roi_data - grand_mean
            else:
                roi_norm = roi_data

            roi_betas_list[roinum].append(roi_norm.astype(np.float32))

        ntrials_total += betas_2d.shape[1]
        del betas_2d
        gc.collect()

    # Concatenate across sessions: each ROI → (nvox, ntrials_used)
    roiBetas = {}
    for roinum in all_rois_needed:
        roiBetas[roinum] = np.concatenate(roi_betas_list[roinum], axis=1)
    del roi_betas_list
    gc.collect()

    # Crop splitImgTrials and subDesign/imgNum to trials we actually have betas for
    # NOTE: for subs with <40 sessions, only the first nsessions*750 trials are used
    ntrials_used = roiBetas[all_rois_needed[0]].shape[1]
    splitImgTrials = splitImgTrials[:, :ntrials_used]
    subDesign_used = subDesign[:ntrials_used]

    # -------------------------------------------------------------------
    # For each visualRegion group separately (like outer loop in MATLAB)
    # -------------------------------------------------------------------
    for visualRegion in visualRegions:
        print(f"\n=== Visual Region group: {visualRegion} ===")
        rois = roi_indices[visualRegion]  # e.g., [0,1] for V1v+V1d

        # -----------------------------------------------------
        # Load pRF samples HDF5 generated by prf_sample_model
        # -----------------------------------------------------
        prf_file = os.path.join(
            PRFSAMPLE_DIR, f"prfSampleStim_v{visualRegion}_sub{isub}.h5"
        )
        if not os.path.exists(prf_file):
            print(f"  ERROR: pRF sample file not found: {prf_file}")
            continue

        with h5py.File(prf_file, "r") as fprf:
            allImgs = fprf["allImgs"][:].astype(np.int32)
            trial_imgids = fprf["trial_imgids"][:].astype(np.int32)  # same as subDesign order
            numLevels = int(fprf.attrs["numLevels"])
            numOrientations = int(fprf.attrs["numOrientations"])

            # Map image ID → row index in prfSample arrays
            imgid_to_row = {img: i for i, img in enumerate(allImgs)}

            # imgNum: trial → index into allImgs (like MATLAB ismember(subDesign, allImgs))
            imgNum = np.full_like(subDesign_used, fill_value=-1, dtype=np.int32)
            for i, img_id in enumerate(subDesign_used):
                if img_id in imgid_to_row:
                    imgNum[i] = imgid_to_row[img_id]
            # logical imgTrials equivalent handled later via masks

            # ----------------- containers per ROI -----------------
            nvox = {}
            r2 = {}
            r2ori = {}
            r2split = {}
            r2oriSplit = {}
            rssSplit = {}
            rssOriSplit = {}
            pearsonR = {}
            pearsonRori = {}
            voxCoef = {}
            voxOriCoef = {}
            voxPredOriCoef = {}
            voxOriPredOriCoef = {}
            voxResidOriCoef = {}
            voxOriResidOriCoef = {}
            voxPredOriR2 = {}
            voxResidOriR2 = {}
            voxOriPredOriR2 = {}
            voxOriResidOriR2 = {}
            sessBetas = {}
            sessStdBetas = {}
            voxCoefCorr = {}
            voxOriCoefCorr = {}
            voxCoefCorrWithConst = {}
            voxOriCoefCorrWithConst = {}
            roiPrf = {}

            # Load roiPrf from HDF5 (optional but matches MATLAB output)
            if "prfParam" in fprf:
                for roinum in rois:
                    roiPrf[roinum] = {
                        "ecc": fprf[f"prfParam/ecc/roi_{roinum}"][:],
                        "ang": fprf[f"prfParam/ang/roi_{roinum}"][:],
                        "sz":  fprf[f"prfParam/sz/roi_{roinum}"][:],
                        "x":   fprf[f"prfParam/x/roi_{roinum}"][:],
                        "y":   fprf[f"prfParam/y/roi_{roinum}"][:],
                    }

            # -----------------------------------------------------
            # Main loop over ROIs
            # -----------------------------------------------------
            for roinum in rois:
                print(f"  ROI {roinum} ({roi_names[roinum]})")

                # pRF samples: (nImages, nVox, numLevels+2 / numLevels×numOrientations)
                prfLev = fprf[f"prfSampleLev/roi_{roinum}"][:]       # float32
                prfLevOri = fprf[f"prfSampleLevOri/roi_{roinum}"][:]  # float32

                Y = roiBetas[roinum]  # (nvox, ntrials_used)
                L = Y.shape[0]
                nvox[roinum] = L

                maxNumTrials = splitImgTrials.sum(axis=1).max()

                # Allocate per-ROI arrays (matching MATLAB shapes)
                voxOriResidual = np.full((nsplits, L, maxNumTrials), np.nan, dtype=np.float32)
                voxResidual = np.full((nsplits, L, maxNumTrials), np.nan, dtype=np.float32)

                # coefficient arrays (nsplits × L × predictors)
                voxCoefSplit = np.zeros((nsplits, L, numLevels + 1 + 2), dtype=np.float32)
                voxOriCoefSplit = np.zeros((nsplits, L, numLevels * numOrientations + 1 + 2),
                                           dtype=np.float32)
                voxPredOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxOriPredOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxResidOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxOriResidOriCoefSplit = np.zeros_like(voxOriCoefSplit)

                r2WithinSplit = np.zeros((nsplits, L), dtype=np.float32)
                r2oriWithinSplit = np.zeros((nsplits, L), dtype=np.float32)
                voxOriResidOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxOriPredOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxResidOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxPredOriR2Split = np.zeros((nsplits, L), dtype=np.float32)

                r2splitVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                r2OrisplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                rssSplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float64)
                rssOrisplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float64)
                pearsonRoriVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                pearsonRVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)

                sessBetasSplit = np.zeros((nsplits, L), dtype=np.float32)
                sessStdBetasSplit = np.zeros((nsplits, L), dtype=np.float32)

                # --------------- First parfor: within-split regressions ---------------
                for isplit in range(nsplits):
                    imgTrials = (splitImgTrials[isplit, :] > 0)
                    numTrials = int(imgTrials.sum())
                    if numTrials == 0:
                        continue

                    sessBetasSplit[isplit, :] = Y[:, imgTrials].mean(axis=1)
                    sessStdBetasSplit[isplit, :] = Y[:, imgTrials].std(axis=1, ddof=0)

                    trial_img_idx = imgNum[imgTrials]
                    # guard: some trials might have imgNum == -1 (no prf sample)
                    valid_mask = (trial_img_idx >= 0)
                    if not np.all(valid_mask):
                        # Drop unmatched trials from both Y and predictors
                        imgTrials_sanitized = imgTrials.copy()
                        imgTrials_sanitized[imgTrials] = valid_mask
                        imgTrials = imgTrials_sanitized
                        trial_img_idx = imgNum[imgTrials]
                        numTrials = int(imgTrials.sum())
                        if numTrials == 0:
                            continue

                    for ivox in range(L):
                        voxBetas = Y[ivox, imgTrials].astype(np.float64)  # (numTrials,)

                        # --- pRF level samples ---
                        voxPrfSample = prfLev[trial_img_idx, ivox, :]  # (numTrials, numLevels+2)
                        # add constant predictor
                        Xlev = np.concatenate(
                            [voxPrfSample, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )  # (numTrials, numLevels+3)
                        coef_lev = lstsq_coeff(Xlev, voxBetas)
                        voxCoefSplit[isplit, ivox, :] = coef_lev

                        # --- pRF orientation samples ---
                        voxPrfOriSample = prfLevOri[trial_img_idx, ivox, :, :]  # (numTrials, numLevels, numOrient)
                        Xori_base = voxPrfOriSample.reshape(numTrials, numLevels * numOrientations)

                        # add lowest & highest SF (taken from Xlev cols end-2:end-1)
                        low_high = Xlev[:, -3:-1]  # those are original low/high SF predictors
                        Xori_ext = np.concatenate([Xori_base, low_high], axis=1)

                        # add constant
                        Xori = np.concatenate(
                            [Xori_ext, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )  # (numTrials, numLevels*numOrient+3)
                        coef_ori = lstsq_coeff(Xori, voxBetas)
                        voxOriCoefSplit[isplit, ivox, :] = coef_ori

                        # --- regress vignetting model prediction with orientation model ---
                        voxPred = Xlev @ coef_lev  # vignetting model prediction
                        coef_predOri = lstsq_coeff(Xori, voxPred)
                        voxPredOriCoefSplit[isplit, ivox, :] = coef_predOri

                        # --- regress full orientation model prediction with orientation model ---
                        voxOriPred = Xori @ coef_ori
                        coef_oriPredOri = lstsq_coeff(Xori, voxOriPred)
                        voxOriPredOriCoefSplit[isplit, ivox, :] = coef_oriPredOri

                        # --- residuals within split ---
                        voxOriResidual_vec = voxBetas - Xori @ coef_ori
                        voxResidual_vec = voxBetas - Xlev @ coef_lev

                        voxOriResidual[isplit, ivox, :numTrials] = voxOriResidual_vec.astype(np.float32)
                        voxResidual[isplit, ivox, :numTrials] = voxResidual_vec.astype(np.float32)

                        # regress residuals of vignetting model with full model
                        coef_residOri = lstsq_coeff(Xori, voxResidual_vec)
                        voxResidOriCoefSplit[isplit, ivox, :] = coef_residOri

                        # regress residuals of orientation model with orientation model
                        coef_oriResidOri = lstsq_coeff(Xori, voxOriResidual_vec)
                        voxOriResidOriCoefSplit[isplit, ivox, :] = coef_oriResidOri

                        # R² within split
                        r2WithinSplit[isplit, ivox] = rsquared(
                            voxResidual_vec, Y[ivox, imgTrials]
                        )
                        r2oriWithinSplit[isplit, ivox] = rsquared(
                            voxOriResidual_vec, Y[ivox, imgTrials]
                        )

                        # residuals of full orientation model after oriResidOri fit
                        oriResidOriResidual = voxBetas - (Xori @ coef_oriResidOri)
                        voxOriResidOriR2Split[isplit, ivox] = rsquared(
                            oriResidOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of full orientation model after oriPredOri fit
                        oriPredOriResidual = voxBetas - (Xori @ coef_oriPredOri)
                        voxOriPredOriR2Split[isplit, ivox] = rsquared(
                            oriPredOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of constrained model (predicted residuals of vignetting model)
                        residOriResidual = voxBetas - (Xori @ coef_residOri)
                        voxResidOriR2Split[isplit, ivox] = rsquared(
                            residOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of constrained model (prediction of constrained model)
                        predOriResidual = voxBetas - (Xori @ coef_predOri)
                        voxPredOriR2Split[isplit, ivox] = rsquared(
                            predOriResidual, Y[ivox, imgTrials]
                        )

                # --------------- Second parfor: cross-split metrics ---------------
                for isplit in range(nsplits):
                    imgTrials = (splitImgTrials[isplit, :] > 0)
                    numTrials = int(imgTrials.sum())
                    if numTrials == 0:
                        continue

                    trial_img_idx = imgNum[imgTrials]
                    valid_mask = (trial_img_idx >= 0)
                    if not np.all(valid_mask):
                        imgTrials_sanitized = imgTrials.copy()
                        imgTrials_sanitized[imgTrials] = valid_mask
                        imgTrials = imgTrials_sanitized
                        trial_img_idx = imgNum[imgTrials]
                        numTrials = int(imgTrials.sum())
                        if numTrials == 0:
                            continue

                    Xlev = prfLev[trial_img_idx, :, :]  # (numTrials, L, numLevels+2)
                    Xori_all = prfLevOri[trial_img_idx, :, :, :]  # (numTrials, L, numLevels, numOrient)

                    for ivox in range(L):
                        voxBetas = Y[ivox, imgTrials].astype(np.float64)  # (numTrials,)

                        # design matrices for this voxel & split
                        Xlev_vox = Xlev[:, ivox, :]  # (numTrials, numLevels+2)
                        Xlev_with_const = np.concatenate(
                            [Xlev_vox, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )

                        Xori_vox = Xori_all[:, ivox, :, :].reshape(
                            numTrials, numLevels * numOrientations
                        )
                        low_high = Xlev_with_const[:, -3:-1]
                        Xori_ext = np.concatenate([Xori_vox, low_high], axis=1)
                        Xori_with_const = np.concatenate(
                            [Xori_ext, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )

                        for nextSplit in range(nsplits):
                            coef_lev_next = voxCoefSplit[nextSplit, ivox, :]
                            coef_ori_next = voxOriCoefSplit[nextSplit, ivox, :]

                            voxResidualSplit_vec = voxBetas - Xlev_with_const @ coef_lev_next
                            voxOriResidualSplit_vec = voxBetas - Xori_with_const @ coef_ori_next

                            r2splitVox[isplit, ivox, nextSplit] = rsquared(
                                voxResidualSplit_vec, voxBetas
                            )
                            r2OrisplitVox[isplit, ivox, nextSplit] = rsquared(
                                voxOriResidualSplit_vec, voxBetas
                            )

                            rssSplitVox[isplit, ivox, nextSplit] = np.sum(
                                voxResidualSplit_vec ** 2
                            )
                            rssOrisplitVox[isplit, ivox, nextSplit] = np.sum(
                                voxOriResidualSplit_vec ** 2
                            )

                            # Pearson correlation between voxBetas and predictions
                            y_pred_ori = Xori_with_const @ coef_ori_next
                            y_pred_lev = Xlev_with_const @ coef_lev_next
                            if np.all(np.isfinite(y_pred_ori)) and np.std(y_pred_ori) > 0:
                                pearsonRoriVox[isplit, ivox, nextSplit] = np.corrcoef(
                                    voxBetas, y_pred_ori
                                )[0, 1]
                            else:
                                pearsonRoriVox[isplit, ivox, nextSplit] = np.nan

                            if np.all(np.isfinite(y_pred_lev)) and np.std(y_pred_lev) > 0:
                                pearsonRVox[isplit, ivox, nextSplit] = np.corrcoef(
                                    voxBetas, y_pred_lev
                                )[0, 1]
                            else:
                                pearsonRVox[isplit, ivox, nextSplit] = np.nan

                # store per-ROI results
                sessBetas[roinum] = sessBetasSplit
                sessStdBetas[roinum] = sessStdBetasSplit

                voxPredOriR2[roinum] = voxPredOriR2Split
                voxResidOriR2[roinum] = voxResidOriR2Split
                voxOriPredOriR2[roinum] = voxOriPredOriR2Split
                voxOriResidOriR2[roinum] = voxOriResidOriR2Split

                r2ori[roinum] = r2oriWithinSplit
                r2[roinum] = r2WithinSplit

                voxResidOriCoef[roinum] = voxResidOriCoefSplit
                voxOriResidOriCoef[roinum] = voxOriResidOriCoefSplit
                voxOriCoef[roinum] = voxOriCoefSplit
                voxCoef[roinum] = voxCoefSplit
                voxPredOriCoef[roinum] = voxPredOriCoefSplit
                voxOriPredOriCoef[roinum] = voxOriPredOriCoefSplit

                r2split[roinum] = r2splitVox
                r2oriSplit[roinum] = r2OrisplitVox
                rssOriSplit[roinum] = rssOrisplitVox
                rssSplit[roinum] = rssSplitVox
                pearsonRori[roinum] = pearsonRoriVox
                pearsonR[roinum] = pearsonRVox

                # ---- correlate coefficients between splits (with & without constant) ----
                voxCoefCorr[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxOriCoefCorr[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxCoefCorrWithConst[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxOriCoefCorrWithConst[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)

                for ivox in range(L):
                    # without constant (drop last predictor)
                    coef_lev_no_const = voxCoefSplit[:, ivox, :-1]      # (nsplits, P)
                    coef_ori_no_const = voxOriCoefSplit[:, ivox, :-1]   # (nsplits, P')

                    # with constant
                    coef_lev_all = voxCoefSplit[:, ivox, :]
                    coef_ori_all = voxOriCoefSplit[:, ivox, :]

                    voxCoefCorr[roinum][ivox, :, :] = np.corrcoef(coef_lev_no_const) if coef_lev_no_const.shape[0] > 1 else np.eye(nsplits)
                    voxOriCoefCorr[roinum][ivox, :, :] = np.corrcoef(coef_ori_no_const) if coef_ori_no_const.shape[0] > 1 else np.eye(nsplits)
                    voxCoefCorrWithConst[roinum][ivox, :, :] = np.corrcoef(coef_lev_all) if coef_lev_all.shape[0] > 1 else np.eye(nsplits)
                    voxOriCoefCorrWithConst[roinum][ivox, :, :] = np.corrcoef(coef_ori_all) if coef_ori_all.shape[0] > 1 else np.eye(nsplits)

            # ---------------- wrap into nsd dict (like MATLAB struct) ----------------
            nsd = {
                "voxOriCoefCorrWithConst": voxOriCoefCorrWithConst,
                "voxCoefCorrWithConst": voxCoefCorrWithConst,
                "voxOriCoefCorr": voxOriCoefCorr,
                "voxCoefCorr": voxCoefCorr,
                "r2": r2,
                "r2ori": r2ori,
                "r2split": r2split,
                "r2oriSplit": r2oriSplit,
                "rssOriSplit": rssOriSplit,
                "rssSplit": rssSplit,
                "pearsonRori": pearsonRori,
                "pearsonR": pearsonR,
                "imgNum": imgNum,
                "splitImgTrials": splitImgTrials,
                "voxCoef": voxCoef,
                "voxOriCoef": voxOriCoef,
                "voxPredOriCoef": voxPredOriCoef,
                "voxOriPredOriCoef": voxOriPredOriCoef,
                "voxResidOriCoef": voxResidOriCoef,
                "voxOriResidOriCoef": voxOriResidOriCoef,
                "voxPredOriR2": voxPredOriR2,
                "voxOriPredOriR2": voxOriPredOriR2,
                "voxResidOriR2": voxResidOriR2,
                "voxOriResidOriR2": voxOriResidOriR2,
                "sessBetas": sessBetas,
                "sessStdBetas": sessStdBetas,
                "roiInd": roiInd,
            }

            # zscore suffix for filename
            if to_zscore == 1:
                zscore_str = "_zscore"
            elif to_zscore == 2:
                zscore_str = "_zeroMean"
            elif to_zscore == 3:
                zscore_str = "_equalStd"
            elif to_zscore == 4:
                zscore_str = "_zeroROImean"
            else:
                zscore_str = ""

            out_name = f"regressPrfSplit_session_v{visualRegion}_sub{isub}{zscore_str}.pkl"
            out_path = os.path.join(SAVE_DIR, out_name)
            print(f"  Saving results to {out_path}")
            with open(out_path, "wb") as f_out:
                pickle.dump(
                    {
                        "nsd": nsd,
                        "numLevels": numLevels,
                        "numOrientations": numOrientations,
                        "rois": rois,
                        "nvox": nvox,
                        "roiPrf": roiPrf,
                        "nsplits": nsplits,
                        "roi_names": roi_names,
                    },
                    f_out,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    t1_global = time.time()
    print(f"\nDone regress_prf_split_expand in {t1_global - t0_global:.1f} s.")


# -----------------------------------------------------------------------
# Example call (like MATLAB: regressPrfSplit_expand(isubject, 1, 0))
# -----------------------------------------------------------------------
regress_prf_split_expand(isub=3, visualRegions=[1], to_zscore=0)


ModuleNotFoundError: No module named 'nibabel'

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

# --------------------------------------------------
# Mount Google Drive (safe)
# --------------------------------------------------
if not os.path.ismount("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

# --------------------------------------------------
# Load the pickle file
# --------------------------------------------------
file_path = "/content/drive/MyDrive/V1_Drift/repDrift_expand/regressPrfSplit_session_v1_sub3.pkl"
assert os.path.exists(file_path), f"File not found: {file_path}"
print(f"\nLoading {file_path} ...")

with open(file_path, "rb") as f:
    data = pickle.load(f)

nsd = data["nsd"]
rois = data["rois"]
roi_names = data["roi_names"]
nsplits = data["nsplits"]
numLevels = data["numLevels"]
numOrientations = data["numOrientations"]
nvox = data["nvox"]

print("\n=== Top-level metadata ===")
print("rois:", rois)
print("roi_names:", roi_names)
print("nsplits:", nsplits)
print("numLevels:", numLevels)
print("numOrientations:", numOrientations)
print("nvox:", nvox)

# ----------------------------
# Helpers
# ----------------------------
def finite_stats(arr, quantiles=(0.05, 0.5, 0.95)):
    """Return stats over finite values only (excludes NaN/±Inf)."""
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    n = arr.size
    n_fin = int(finite.sum())
    n_nan = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    pct_nan = 100.0 * n_nan / n if n else 0.0
    pct_inf = 100.0 * n_inf / n if n else 0.0

    if n_fin == 0:
        return {
            "n": n, "n_fin": 0, "n_nan": n_nan, "n_inf": n_inf,
            "pct_nan": pct_nan, "pct_inf": pct_inf,
            "mean": None, "std": None, "min": None, "max": None, "q": None
        }

    vals = arr[finite].astype(np.float64, copy=False)
    qvals = np.quantile(vals, quantiles) if len(vals) > 0 else [None] * len(quantiles)
    return {
        "n": n, "n_fin": n_fin, "n_nan": n_nan, "n_inf": n_inf,
        "pct_nan": pct_nan, "pct_inf": pct_inf,
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "q": [float(x) for x in qvals]
    }

def print_stats_line(name, arr, extra=None):
    s = finite_stats(arr)
    q = s["q"]
    if q and s["n_fin"]:
        qtxt = f" | q05={q[0]:.4g}, q50={q[1]:.4g}, q95={q[2]:.4g}"
    else:
        qtxt = " | q05/50/95=NA"

    mean_str = f"{s['mean']:.4g}" if s["mean"] is not None else "NA"
    std_str = f"{s['std']:.4g}" if s["std"] is not None else "NA"
    min_str = f"{s['min']:.4g}" if s["min"] is not None else "NA"
    max_str = f"{s['max']:.4g}" if s["max"] is not None else "NA"

    print(
        f"{name:<35} shape={arr.shape}  "
        f"fin={s['n_fin']}/{s['n']}  NaN={s['pct_nan']:.2f}%  Inf={s['pct_inf']:.2f}%  "
        f"mean={mean_str}  std={std_str}  min={min_str}  max={max_str}"
        f"{qtxt}"
        + (f"  {extra}" if extra else "")
    )

def offdiag_mask(shape):
    """Boolean mask for off-diagonal entries in (S,S) matrices."""
    assert len(shape) >= 2 and shape[-1] == shape[-2]
    S = shape[-1]
    m = np.ones((S, S), dtype=bool)
    np.fill_diagonal(m, False)
    return m

# --------------------------------------------------
# Quick stats for all nsd keys
# --------------------------------------------------
print("\n=== nsd keys (types) ===")
for key, val in nsd.items():
    if isinstance(val, dict):
        print(f"{key:<25} dict with ROIs: {list(val.keys())}")
    else:
        print_stats_line(key, np.asarray(val))

# --------------------------------------------------
# Detailed per-ROI stats for key metrics
# --------------------------------------------------
keys_main = [
    "r2", "r2ori",
    "r2split", "r2oriSplit",
    "rssSplit", "rssOriSplit",
    "pearsonR", "pearsonRori",
    "voxCoefCorr", "voxOriCoefCorr",
    "voxCoefCorrWithConst", "voxOriCoefCorrWithConst",
]

for roi in rois:
    print(f"\n================ ROI {roi} ({roi_names[roi]}) ================")

    for k in keys_main:
        if k not in nsd:
            continue
        val = nsd[k]
        if not isinstance(val, dict) or roi not in val:
            continue

        arr = np.asarray(val[roi])

        # 3D: (S, L, S) e.g. r2split, pearsonR
        if arr.ndim == 3 and arr.shape[0] == arr.shape[2]:
            S, L, _ = arr.shape
            print_stats_line(k, arr)

            # diagonal (same split train==test)
            diag_vals = np.empty((L, S), dtype=float)
            for s in range(S):
                diag_vals[:, s] = arr[s, :, s]
            print_stats_line(k + " (diag over splits)", diag_vals)

            # off-diagonal (different splits)
            off_list = []
            for s in range(S):
                for t in range(S):
                    if s == t:
                        continue
                    off_list.append(arr[s, :, t])
            off_vals = np.stack(off_list, axis=1) if off_list else np.empty((L, 0))
            print_stats_line(k + " (offdiag over splits)", off_vals)

        # 3D: (V, S, S) e.g. voxCoefCorr
        elif arr.ndim == 3 and arr.shape[1] == arr.shape[2]:
            V, S, _ = arr.shape
            print_stats_line(k, arr)

            m_off = offdiag_mask((S, S))
            off_vals = arr[:, m_off].reshape(V, -1)
            print_stats_line(k + " (offdiag only)", off_vals)

            diag_vals = np.diagonal(arr, axis1=1, axis2=2)  # (V, S)
            print_stats_line(k + " (diag only)", diag_vals)

        else:
            # 1D/2D metrics (r2, r2ori, etc.)
            print_stats_line(k, arr)

# --------------------------------------------------
# Example: coefficient-correlation heatmap for one voxel
# --------------------------------------------------
roi_example = rois[0]          # e.g. first ROI (0 -> V1v)
ivox = 100                      # voxel index to display
key_corr = "voxCoefCorr"

if key_corr in nsd and roi_example in nsd[key_corr]:
    corr = nsd[key_corr][roi_example][ivox]  # (S, S)
    print(f"\nVoxel {ivox} in ROI {roi_example} ({roi_names[roi_example]})")
    print("  corr shape:", corr.shape)

    s_all = finite_stats(corr)
    print(f"  mean r (finite-only) = {s_all['mean'] if s_all['mean'] is not None else 'NA'}")

    if corr.shape[0] >= 6:
        row = corr[5, :]
        row_fin = row[np.isfinite(row)]
        print("  Row 6 finite mean:", np.mean(row_fin) if row_fin.size else 'NA')
        print("  Row 6 values (first 10):", np.round(row[:10], 3))

    masked = np.ma.masked_invalid(corr)
    plt.figure(figsize=(5, 4))
    im = plt.imshow(masked, interpolation="nearest")
    plt.title(f"ROI {roi_example} — voxel {ivox} coef corr (masked invalid)")
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()
else:
    print(f"\nNo '{key_corr}' data available for ROI {roi_example}.")



Attributes → num_levels: 7, num_orientations: 8
Loaded allImgs: shape = (975,), dtype = int32
Loaded rois: shape = (2,), dtype = int32

Loading PRF samples:

ROI 0:
  prfSampleLev     shape = (10000, 594, 9), dtype = float64
  prfSampleLevOri  shape = (10000, 594, 9, 8), dtype = float64

ROI 1:
  prfSampleLev     shape = (10000, 756, 9), dtype = float64
  prfSampleLevOri  shape = (10000, 756, 9, 8), dtype = float64


# Optimized

In [ ]:
import numpy as np
import nibabel as nib
import scipy.io as sio
import os
import h5py
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed


# --------------------------------------------------
# Utility: Compute R² between residual and original
# --------------------------------------------------


def rsquared(Xresid, Xorig):
    return 1 - np.sum((Xresid) ** 2) / np.sum((Xorig - np.mean(Xorig)) ** 2)


# ----------------------------------------------------------------------------
# Main Function: Perform voxel-wise regression of PRF features on NSD betas
# isub: subject index (1-based like MATLAB)
# visual_regions: list of visual regions (e.g., [1, 2, 3])
# to_zscore: normalization method (0: none, 1: z-score, 2: mean, etc.)
# ----------------------------------------------------------------------------


def regress_prf_split_expand(isub, visual_regions=[1], to_zscore=0):
    # -------------------------------
    # Setup & File Paths
    # -------------------------------

    # Number of sessions per subject
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]
    nsessions = nsessions_sub[isub - 1]
    nsplits = nsessions  # One split per session

    # bandpass info
    bandpass = 1
    band_min = 1
    band_max = 7

    # Set data folders
    prf_folder = r"C:\nsd\prfsample_expand"
    save_folder = r"C:\nsd\repDrift_expand/"
    nsd_folder = r"D:\NSD"
    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    visual_rois_file = os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz")

    # Load visual ROI mask (NIfTI)
    vis_roi_img = nib.load(visual_rois_file)
    vis_roi_data = vis_roi_img.get_fdata().flatten()  # Flatten for voxel indexing

    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]

    # ------------------------------------------------------------
    # Define visual region mapping (visualRegion → ROI indices)
    # ------------------------------------------------------------

    # Each visualRegion value corresponds to a group of ROIs:
    #   1 → [V1v, V1d] (ROI indices 0 and 1)
    #   2 → [V2v, V2d] (ROI indices 2 and 3)
    #   3 → [V3v, V3d] (ROI indices 4 and 5)
    #   4 → [hV4]      (ROI index 6)
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],  # hV4
    }

    # Load NSD experiment design
    nsd_design = sio.loadmat(os.path.join(nsd_folder, "nsd_expdesign.mat"))
    master_ordering = nsd_design["masterordering"].squeeze() - 1  # MATLAB 1-indexed
    sub_design = nsd_design["subjectim"][isub - 1, master_ordering].squeeze()

    # ------------------------------------------
    # Split trial matrix (1 row per split)
    # ------------------------------------------

    ntrials = len(master_ordering)
    trials_per_session = 10 * 75  # Each session: 10 blocks × 75 trials
    split_img_trials = np.zeros((nsplits, ntrials), dtype=bool)

    itrial = 0
    for isplit in range(nsplits):  # Define split_img_trials once
        split_img_trials[isplit, itrial : itrial + trials_per_session] = True
        itrial += trials_per_session

    # ------------------------------------------
    # Loop over Visual Regions
    # ------------------------------------------

    for (
        visual_region
    ) in visual_regions:  # # Outer loop: per visual ROI (e.g., V1v, V1d...)
        rois = roi_map.get(visual_region, [])
        print(f"Processing visual region: {visual_region}")

        # --- Load PRF sample features for this region ---
        prf_file = os.path.join(
            prf_folder,
            f"prfSampleStim_v{visual_region}_sub{isub}.h5",  # Load the PRF energy for this region
        )
        with h5py.File(prf_file, "r") as f:
            # Load scalar attributes
            num_levels = int(f.attrs["numLevels"])
            num_orientations = int(f.attrs["numOrientations"])
            roi_prf = None  # Optional — only used for saving, not needed for regression

            # Load global image list and ROI list
            all_imgs = np.array(f["allImgs"]).squeeze()
            rois = np.array(f["rois"]).squeeze()

            # Load prfSampleLev and prfSampleLevOri into Python dictionaries
            prf_sample_lev = {}
            prf_sample_lev_ori = {}

            for roinum in rois:
                prf_sample_lev[roinum] = np.array(
                    f[f"prfSampleLev/roi_{roinum}"]
                )  # shape: (n_imgs, n_vox, num_levels + 2)
                prf_sample_lev_ori[roinum] = np.array(
                    f[f"prfSampleLevOri/roi_{roinum}"]
                )  # shape: (n_imgs, n_vox, num_levels + 2, num_orientations)

        # --- Init containers for each ROI ---
        roi_betas = {roinum: [] for roinum in range(len(rois))}  # beta responses
        roi_ind = {}  # voxel indices per ROI

        # --- Load and normalize beta data per session ---
        for isession in range(1, nsessions + 1):  # For each session
            beta_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii")
            beta_img = nib.load(beta_file)
            beta_data = beta_img.get_fdata().astype(np.float64)
            beta_data = beta_data / 300  # Convert to percent signal change
            beta_data = beta_data.reshape(-1, beta_data.shape[-1])  # Flatten 4D to 2D

            for roinum, iroi in enumerate(
                rois
            ):  # For each ROI within this visual region
                vox_idx = vis_roi_data == iroi
                roi_ind[roinum] = np.where(vox_idx)[0]
                roi_voxels = beta_data[vox_idx, :]  # Get betas for this ROI

                # --- Apply normalization method ---
                if to_zscore == 1:
                    norm_voxels = zscore(roi_voxels, axis=1, ddof=0)
                elif to_zscore == 2:
                    norm_voxels = roi_voxels - roi_voxels.mean(axis=1, keepdims=True)
                elif to_zscore == 3:
                    mean_ = roi_voxels.mean(axis=1, keepdims=True)
                    norm_voxels = mean_ + zscore(roi_voxels - mean_, axis=1, ddof=0)
                elif to_zscore == 4:
                    norm_voxels = roi_voxels - roi_voxels.mean()
                else:
                    norm_voxels = roi_voxels  # No normalization

                roi_betas[roinum].append(norm_voxels)

        # --- Concatenate sessions along time axis ---
        for roinum in roi_betas:
            roi_betas[roinum] = np.concatenate(roi_betas[roinum], axis=1)

        # --- Match presented images to full image set ---
        img_trials_mask, img_num = (
            np.isin(sub_design, all_imgs, assume_unique=True),
            None,
        )
        img_num = np.searchsorted(
            all_imgs, sub_design[img_trials_mask]
        )  # Shape: (n_trials,)

        # --- Ensure trial counts match for this subject ---
        for roinum in roi_betas:
            roi_betas[roinum] = roi_betas[roinum][:, : split_img_trials.shape[1]]
        split_img_trials = split_img_trials[:, : roi_betas[roinum].shape[1]]
        img_num = img_num[: roi_betas[roinum].shape[1]]

        max_num_trials = split_img_trials.sum(axis=1).max()  # Used for array size later

        # Prepare containers per ROI
        nsd = {}
        nvox = {}
        r2 = {}
        r2ori = {}
        r2split = {}
        r2ori_split = {}
        rss_split = {}
        rss_ori_split = {}
        pearson_r = {}
        pearson_rori = {}
        sess_betas = {}
        sess_std_betas = {}

        vox_coef = {}
        vox_ori_coef = {}
        vox_pred_ori_coef = {}
        vox_ori_pred_ori_coef = {}
        vox_resid_ori_coef = {}
        vox_ori_resid_ori_coef = {}

        vox_pred_ori_r2 = {}
        vox_resid_ori_r2 = {}
        vox_ori_pred_ori_r2 = {}
        vox_ori_resid_ori_r2 = {}

        vox_coef_corr = {}
        vox_ori_coef_corr = {}
        vox_coef_corr_with_const = {}
        vox_ori_coef_corr_with_const = {}

        # ------------------------------------------------------------
        # Voxel-wise Regression for Each ROI and Each Split
        # ------------------------------------------------------------
        for roinum in range(len(rois)):
            print(f"  → Processing ROI {roinum} (ID {rois[roinum]})")
            nvox[roinum] = roi_betas[roinum].shape[0]
            L = nvox[roinum]

            # Allocate arrays for this ROI
            shape_lev = num_levels + 1
            shape_ori = num_levels * num_orientations + 1

            # Initialize coefficient and residual matrices per voxel per split
            vox_coef[roinum] = np.zeros((nsplits, L, shape_lev))
            vox_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))

            vox_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            r2[roinum] = np.zeros((nsplits, L))
            r2ori[roinum] = np.zeros((nsplits, L))
            sess_betas[roinum] = np.zeros((nsplits, L))
            sess_std_betas[roinum] = np.zeros((nsplits, L))

            # Loop over splits
            for isplit in range(nsplits):
                img_trials_split = split_img_trials[isplit]
                trial_idx = np.where(img_trials_split)[0]

                betas_split = roi_betas[roinum][:, trial_idx]  # shape: voxels × trials
                sess_betas[roinum][isplit] = np.mean(betas_split, axis=1)
                sess_std_betas[roinum][isplit] = np.std(betas_split, axis=1)

                # ---------------------------
                # Precompute X matrix for all voxels
                # ---------------------------
                X_all_vox = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # (n_trials, n_voxels, num_levels+2)
                n_trials, n_voxels, n_feats = X_all_vox.shape

                # Precompute design matrices for all voxels
                X_ori_all = prf_sample_lev_ori[roinum][img_num[trial_idx], :, :, :]  # (n_trials, n_voxels, levels, orientations)
                X_ori_flat = X_ori_all.reshape(X_ori_all.shape[0], X_ori_all.shape[1], -1)  # (n_trials, n_voxels, lev*ori)

                # Get extra features: lowest SF and highest SF
                X_basic = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # (n_trials, n_voxels, num_levels+2)

                n_trials, n_voxels, _ = X_basic.shape
                n_features_ori = X_ori_flat.shape[-1] + 2 + 1  # +2 for lowest, highest SF, +1 for constant

                # Prebuild full design matrix: (voxels, trials, features)
                X_ori_stack = np.zeros((n_voxels, n_trials, n_features_ori))
                for ivox in range(n_voxels):
                    X_ori_stack[ivox, :, :-3] = X_ori_flat[:, ivox, :]
                    X_ori_stack[ivox, :, -3] = X_basic[:, ivox, -2]  # lowest SF
                    X_ori_stack[ivox, :, -2] = X_basic[:, ivox, -3]  # highest SF
                    X_ori_stack[ivox, :, -1] = 1  # constant

                # Flatten design matrix to shape (n_trials, n_voxels, n_feats)
                X_stack = np.zeros((n_trials, n_voxels, n_feats + 1))  # +1 for constant
                X_stack[:, :, :-1] = X_all_vox
                X_stack[:, :, -1] = 1  # Add constant column

                # Reshape to (n_trials, n_voxels * (features + 1))
                X_stack = X_stack.transpose(1, 0, 2)  # (n_voxels, n_trials, features+1)

                # ---------------------------
                # Solve all voxels at once
                # ---------------------------
                for ivox in range(L):
                    X = X_stack[ivox]  # (n_trials, features+1)
                    y = betas_split[ivox, :]  # (n_trials,)

                    # Solve least squares
                    coef = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_coef[roinum][isplit, ivox, :] = coef

                    pred = X @ coef
                    residual = y - pred
                    r2[roinum][isplit, ivox] = rsquared(residual, y)

                    # ---------- PRF: Orientation-Level Model ----------
                    X = X_ori_stack[ivox, :, :]  # (n_trials, features)
                    y = betas_split[ivox, :]

                    coef_ori = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_ori_coef[roinum][isplit, ivox, :] = coef_ori

                    pred_ori = X @ coef_ori
                    residual_ori = y - pred_ori
                    r2ori[roinum][isplit, ivox] = rsquared(residual_ori, y)

                    # ---------- Residual regressions ----------
                    # Predict orientation residuals with orientation model
                    vox_ori_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual_ori, rcond=None
                    )[0]
                    pred_ori_resid = np.dot(
                        X_ori, vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    )  # pred_ori_resid = X_ori @ vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_ori_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual_ori - pred_ori_resid, y
                    )

                    # Predict basic residuals with orientation model
                    vox_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual, rcond=None
                    )[0]
                    pred_resid_ori = np.dot(
                        X_ori, vox_resid_ori_coef[roinum][isplit, ivox, :]
                    )  # pred_resid_ori = X_ori @ vox_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual - pred_resid_ori, y
                    )

                    # Predict prediction with orientation model
                    pred_from_basic = pred
                    vox_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_from_basic, rcond=None
                    )[0]
                    pred_pred_ori = np.dot(
                        X_ori, vox_pred_ori_coef[roinum][isplit, ivox, :]
                    )
                    vox_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_from_basic - pred_pred_ori, y
                    )

                    # Predict full orientation prediction with same
                    vox_ori_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_ori, rcond=None
                    )[0]
                    pred_ori_pred_ori = np.dot(
                        X_ori, vox_ori_pred_ori_coef[roinum][isplit, ivox, :]
                    )
                    vox_ori_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_ori - pred_ori_pred_ori, y
                    )
            # Containers for cross-split comparisons
            r2split[roinum] = np.zeros((nsplits, L, nsplits))
            r2ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_r[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_rori[roinum] = np.zeros((nsplits, L, nsplits))

            # ----------------------------
            # Loop: isplit (target session)
            # ----------------------------
            for isplit in range(nsplits):
                trial_idx = np.where(split_img_trials[isplit])[0]
                y = roi_betas[roinum][:, trial_idx]  # shape: (voxels, trials)

                for ivox in range(L):
                    y_vox = y[ivox, :]

                    # Get design matrices for isplit
                    X = prf_sample_lev[roinum][
                        img_num[trial_idx], :, :
                    ]  # shape: (n_trials, n_vox, num_levels+2)
                    X_vox = X[:, ivox, :]  # shape: (n_trials, num_levels+2)
                    X = np.column_stack([X_vox, np.ones(X_vox.shape[0])])

                    X_ori_all = prf_sample_lev_ori[roinum][
                        img_num[trial_idx], :, :, :
                    ]  # (n_trials, n_vox, lev, ori)
                    X_ori_vox = X_ori_all[:, ivox, :, :].reshape(
                        len(trial_idx), -1
                    )  # → (n_trials, lev * ori)
                    X_ori = np.column_stack(
                        [
                            X_ori_vox,
                            X_vox[:, -2],  # lowest SF
                            X_vox[:, -3],  # highest SF
                            np.ones(X_vox.shape[0]),
                        ]
                    )

                    # ----------------------------
                    # Loop: nextSplit (model source)
                    # ----------------------------
                    for next_split in range(nsplits):
                        coef = vox_coef[roinum][next_split, ivox, :]
                        coef_ori = vox_ori_coef[roinum][next_split, ivox, :]

                        # Predict from basic model
                        y_pred = np.dot(X, coef)  # y_pred = X @ coef
                        residual = y_vox - y_pred
                        r2split[roinum][isplit, ivox, next_split] = rsquared(
                            residual, y_vox
                        )
                        rss_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual**2
                        )
                        pearson_r[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred
                        )[0, 1]

                        # Predict from orientation model
                        y_pred_ori = np.dot(
                            X_ori, coef_ori
                        )  # y_pred_ori = X_ori @ coef_ori
                        residual_ori = y_vox - y_pred_ori
                        r2ori_split[roinum][isplit, ivox, next_split] = rsquared(
                            residual_ori, y_vox
                        )
                        rss_ori_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual_ori**2
                        )
                        pearson_rori[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred_ori
                        )[0, 1]

            # --------------------------------------------------------
            # Correlation Between Splits for Each Voxel
            # --------------------------------------------------------
            vox_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))

            for ivox in range(L):
                # Without constant (drop last column)
                coef_mat = vox_coef[roinum][:, ivox, :-1]
                ori_mat = vox_ori_coef[roinum][:, ivox, :-1]

                vox_coef_corr[roinum][ivox] = np.corrcoef(coef_mat, rowvar=False)
                vox_ori_coef_corr[roinum][ivox] = np.corrcoef(ori_mat, rowvar=False)

                # With constant
                coef_mat_full = vox_coef[roinum][:, ivox, :]
                ori_mat_full = vox_ori_coef[roinum][:, ivox, :]

                vox_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    coef_mat_full, rowvar=False
                )
                vox_ori_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    ori_mat_full, rowvar=False
                )

        # -----------------------------------------------------
        # Package Results in Dictionary and Save
        # -----------------------------------------------------
        nsd = {
            "voxOriCoefCorrWithConst": vox_ori_coef_corr_with_const,
            "voxCoefCorrWithConst": vox_coef_corr_with_const,
            "voxOriCoefCorr": vox_ori_coef_corr,
            "voxCoefCorr": vox_coef_corr,
            "r2": r2,
            "r2ori": r2ori,
            "r2split": r2split,
            "r2oriSplit": r2ori_split,
            "rssOriSplit": rss_ori_split,
            "rssSplit": rss_split,
            "pearsonRori": pearson_rori,
            "pearsonR": pearson_r,
            "imgNum": img_num,
            "splitImgTrials": split_img_trials,
            "voxCoef": vox_coef,
            "voxOriCoef": vox_ori_coef,
            "voxPredOriCoef": vox_pred_ori_coef,
            "voxOriPredOriCoef": vox_ori_pred_ori_coef,
            "voxResidOriCoef": vox_resid_ori_coef,
            "voxOriResidOriCoef": vox_ori_resid_ori_coef,
            "voxPredOriR2": vox_pred_ori_r2,
            "voxResidOriR2": vox_resid_ori_r2,
            "voxOriPredOriR2": vox_ori_pred_ori_r2,
            "voxOriResidOriR2": vox_ori_resid_ori_r2,
            "sessBetas": sess_betas,
            "sessStdBetas": sess_std_betas,
            "roiInd": roi_ind,
        }

        # Choose filename suffix based on z-score method
        zscore_str = {
            1: "_zscore",
            2: "_zeroMean",
            3: "_equalStd",
            4: "_zeroROImean",
        }.get(to_zscore, "")

        # Save to .mat
        save_path = os.path.join(
            save_folder,
            f"regressPrfSplit_session_v{visual_region}_sub{isub}{zscore_str}.mat",
        )

        sio.savemat(
            save_path,
            {
                "nsd": nsd,
                "numLevels": num_levels,
                "numOrientations": num_orientations,
                "rois": rois,
                "nvox": nvox,
                "roiPrf": roi_prf,
                "nsplits": nsplits,
            },
        )

        print(f"Saved results to: {save_path}")